In [0]:
# 04_gold_dimensoes_e_fato.py
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver   = spark.table("silver.sop_forecast.vendas_limpo")
df_clusters = spark.table("silver.sop_forecast.clusters_sku")

# ── dim_tempo ──────────────────────────────────────────────
dim_tempo = (
    df_silver.select("ano_mes").distinct()
    .withColumn("ano", F.year("ano_mes"))
    .withColumn("mes", F.month("ano_mes"))
    .withColumn("trimestre", F.quarter("ano_mes"))
    .orderBy("ano_mes")
)
dim_tempo.write.format("delta").mode("overwrite").saveAsTable("gold.sop_forecast.dim_tempo")

# ── dim_loja ───────────────────────────────────────────────
dim_loja = (
    df_silver.select("loja_id").distinct()
    .withColumn("nome_loja", F.concat(F.lit("Loja "), F.col("loja_id")))
    .orderBy("loja_id")
)
dim_loja.write.format("delta").mode("overwrite").saveAsTable("gold.sop_forecast.dim_loja")

# ── dim_item ───────────────────────────────────────────────
dim_item = (
    df_silver.select("item_id").distinct()
    .withColumn("nome_item", F.concat(F.lit("Item "), F.col("item_id")))
    .orderBy("item_id")
)
dim_item.write.format("delta").mode("overwrite").saveAsTable("gold.sop_forecast.dim_item")

# ── fato_vendas_mensal ───────────────────────────────────────
fato_vendas_mensal = (
    df_silver.groupBy("loja_id", "item_id", "ano_mes")
    .agg(F.sum("vendas").alias("vendas_mes"))
)
fato_vendas_mensal.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.sop_forecast.fato_vendas_mensal")

# ── dim_item_loja (bridge: cluster + mix histórico) ────────
volume_total = (
    fato_vendas_mensal.groupBy("loja_id", "item_id")
    .agg(F.sum("vendas_mes").alias("volume_total"))
)

volume_com_cluster = volume_total.join(df_clusters, on=["loja_id", "item_id"], how="left")

w = Window.partitionBy("loja_id", "cluster_id")
dim_item_loja = (
    volume_com_cluster
    .withColumn("volume_cluster", F.sum("volume_total").over(w))
    .withColumn("mix_pct", F.col("volume_total") / F.col("volume_cluster"))
    .select("loja_id", "item_id", "cluster_id", "volume_total", "mix_pct")
)

dim_item_loja.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.sop_forecast.dim_item_loja")

print("Contagens finais:")
print(f"  dim_tempo         : {dim_tempo.count()}")
print(f"  dim_loja          : {dim_loja.count()}")
print(f"  dim_item          : {dim_item.count()}")
print(f"  fato_vendas_mensal: {fato_vendas_mensal.count():,}")
print(f"  dim_item_loja     : {dim_item_loja.count()}")

# validação: soma de mix_pct por loja+cluster deve ser 1.0
validacao = (
    dim_item_loja.groupBy("loja_id", "cluster_id")
    .agg(F.round(F.sum("mix_pct"), 6).alias("soma_mix"))
)
display(validacao)